# Traversal-Based Querying

## Setup

If you haven't already, install the toolkit and dependencies using the [Setup](./00-Setup.ipynb) notebook.

### TraversalBasedRetriever

See [TraversalBasedRetriever](https://github.com/awslabs/graphrag-toolkit/blob/main/docs/lexical-graph/querying.md#traversalbasedretriever).

In [ ]:
%reload_ext dotenv
%dotenv ../.env


import os

from graphrag_toolkit.lexical_graph import set_logging_config
from graphrag_toolkit.lexical_graph import LexicalGraphQueryEngine
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.lexical_graph.storage import VectorStoreFactory

set_logging_config('INFO')

graph_store = GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE'])
vector_store = VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE'])

query_engine = LexicalGraphQueryEngine.for_traversal_based_search(
    graph_store, 
    vector_store,
    streaming=True
)

response = query_engine.query("What are the differences between Neptune Database and Neptune Analytics?")

print(f"""{response.print_response_stream()}

retrieve_ms: {int(response.metadata['retrieve_ms'])}
answer_ms  : {int(response.metadata['answer_ms'])}
total_ms   : {int(response.metadata['total_ms'])}
""")

Based on the search results, the key differences between Neptune Database and Neptune Analytics are:

Neptune Database is a serverless graph database service provided by AWS, designed for optimal scalability and availability. It can scale to 100,000 queries per second and provides multi-AZ high availability and multi-Region deployments. [Source: Amazon Neptune]

In contrast, Neptune Analytics is a memory-optimized graph database engine for analytics. It is designed to enable quick analysis and insights on large graph datasets by storing them in memory. Neptune Analytics provides a library of optimized graph analytic algorithms, low-latency graph queries, and vector search capabilities within graph traversals. [Source: Neptune Analytics]

While Neptune Database is focused on providing a scalable and highly available graph database, Neptune Analytics is optimized for fast analytical processing and insights on graph data. Neptune Analytics can be used to analyze and query graphs in data s

In [11]:
print(f"Graph Store Type: {type(graph_store)}")
print(f"Graph Store: {graph_store}")
print(f"Graph ID: {graph_store.graph_id if hasattr(graph_store, 'graph_id') else 'N/A'}")
print(f"Tenant ID: {graph_store.tenant_id if hasattr(graph_store, 'tenant_id') else 'N/A'}")

Graph Store Type: <class 'graphrag_toolkit.lexical_graph.storage.graph.neptune_graph_stores.NeptuneAnalyticsClient'>
Graph Store: log_formatting=RedactedGraphQueryLogFormatting() tenant_id=TenantId(value=None) graph_id='g-4o5cjg3ix7' config='{}'
Graph ID: g-4o5cjg3ix7
Tenant ID: default_


In [13]:
graph_store.graph_id 

'g-4o5cjg3ix7'

#### Show the context passed to the LLM:

In [2]:
for n in response.source_nodes:
    print(n.text)

{
  "source": "https://docs.aws.amazon.com/neptune-analytics/latest/userguide/what-is-neptune-analytics.html (aws-neptune-docs)",
  "topic": "Neptune Analytics",
  "statements": [
    "Neptune Analytics complements Amazon Neptune Database, a popular managed graph database.",
    "What is Neptune Analytics?",
    "Neptune Analytics is a memory-optimized graph database engine for analytics.",
    "Neptune Analytics stores large graph datasets in memory to enable quick analysis and insights.",
    "Neptune Analytics is an ideal choice for investigatory, exploratory, or data-science workloads that require fast iteration for data, analytical and algorithmic processing, or vector search on graph data.",
    "Neptune Analytics supports a library of optimized graph analytic algorithms, low-latency graph queries, and vector search capabilities within graph traversals."
  ]
}
{
  "source": "https://docs.aws.amazon.com/neptune-analytics/latest/userguide/neptune-analytics-vs-neptune-database.html 

#### Show the underlying results:

In [3]:
import json
for n in response.source_nodes:
    print(json.dumps(n.metadata, indent=2))

{
  "source": {
    "sourceId": "aws::bd6a6b1b:9b34",
    "metadata": {
      "source": "aws-neptune-docs",
      "url": "https://docs.aws.amazon.com/neptune-analytics/latest/userguide/what-is-neptune-analytics.html"
    }
  },
  "topics": [
    {
      "topic": "Neptune Analytics",
      "statements": [
        {
          "statementId": "2bb51038d0846493acc44c650132b2a7",
          "statement": "Neptune Analytics complements Amazon Neptune Database, a popular managed graph database.",
          "facts": [
            "Neptune Analytics COMPLEMENTS Amazon Neptune Database",
            "Amazon Neptune Database TYPE popular managed graph database"
          ],
          "details": "",
          "chunkId": "aws::bd6a6b1b:9b34:693e8f3f",
          "score": 0.65,
          "statement_str": "Neptune Analytics complements Amazon Neptune Database, a popular managed graph database. (details: Neptune Analytics COMPLEMENTS Amazon Neptune Database, Amazon Neptune Database TYPE popular managed gr

#### Visualise the results:

In [4]:
from graphrag_toolkit.lexical_graph.retrieval.model import SearchResult

def get_query_params_for_results(response, include_sources=True, include_facts=True, limit=-1):

    statement_ids = []
    source_params = []
    fact_params = []
    
    nodes = response[:limit] if isinstance(response, list) else response.source_nodes[:limit]
    
    for n in nodes:
        
        search_result = SearchResult.model_validate(n.metadata)
        source_id = search_result.source.sourceId
        
        for topic in search_result.topics:
            
            for statement in topic.statements:
                
                statement_id = statement.statementId
                chunk_id = statement.chunkId
                
                statement_ids.append(statement_id)
                if include_sources:
                    source_params.append({'s': source_id, 'c': chunk_id, 'l': statement_id})
                if include_facts:
                    fact_params.append(statement_id)
                    
    
    query_parameters = { 
        'statement_ids': statement_ids,
        'source_params': source_params,
        'fact_params': fact_params
    }
    
    return query_parameters
    
query_parameters = get_query_params_for_results(response, limit=10)

In [5]:
display_var = '{"__Source__":"url","__Chunk__":"value","__Topic__":"value","__Statement__":"value","__Fact__":"value"}'

In [6]:
%%oc --query-parameters query_parameters -d $display_var -l 20

UNWIND $source_params AS source_params
MATCH p=(s:`__Source__`)<--(c:`__Chunk__`)<--(t:`__Topic__`)<--(l:`__Statement__`)
WHERE id(s) = source_params.s 
    AND id(c) = source_params.c 
    AND id(l) = source_params.l
RETURN p
UNION
MATCH p=(x:`__Source__`)<--(:`__Chunk__`)<--(:`__Topic__`)<--(l:`__Statement__`)<-[:`__SUPPORTS__`]-(:`__Fact__`)-[:`__NEXT__`*0..1]->(:`__Fact__`)-[:`__SUPPORTS__`]->(ll:`__Statement__`)-->(:`__Topic__`)-->(:`__Chunk__`)-->(y:`__Source__`)
WHERE id(l) IN $fact_params
    AND id(ll) IN $fact_params
    AND x <> y
RETURN p
UNION
MATCH p=(l:`__Statement__`)
WHERE id(l) IN $statement_ids
RETURN p

UsageError: Cell magic `%%oc` not found.


In [17]:
import boto3
import json
import os
from urllib.parse import urlparse

# Initialize Neptune client
neptune = boto3.client('neptune-graph', region_name='us-east-1')

# Get the actual graph ID from environment variables
graph_store_url = os.environ['GRAPH_STORE']
graph_id = urlparse(graph_store_url).netloc

print(f"Using graph ID: {graph_id}")

# Your Cypher query
cypher_query = """
UNWIND $source_params AS source_params
MATCH p=(s:`__Source__`)<--(c:`__Chunk__`)<--(t:`__Topic__`)<--(l:`__Statement__`)
WHERE id(s) = source_params.s 
    AND id(c) = source_params.c 
    AND id(l) = source_params.l
RETURN p
UNION
MATCH p=(x:`__Source__`)<--(:`__Chunk__`)<--(:`__Topic__`)<--(l:`__Statement__`)<-[:`__SUPPORTS__`]-(:`__Fact__`)-[:`__NEXT__`*0..1]->(:`__Fact__`)-[:`__SUPPORTS__`]->(ll:`__Statement__`)-->(:`__Topic__`)-->(y:`__Source__`)
WHERE id(l) IN $fact_params
    AND id(ll) IN $fact_params
    AND x <> y
RETURN p
UNION
MATCH p=(l:`__Statement__`)
WHERE id(l) IN $statement_ids
RETURN p
"""

# Execute the query with the actual graph ID
response = neptune.execute_query(
    graphIdentifier=graph_id,
    queryString=cypher_query,
    parameters=query_parameters,
    language='opencypher'
)

# Read the streaming response
payload = response['payload']
data = json.loads(payload.read().decode('utf-8'))

print("Query results:")
print(json.dumps(data, indent=2))

# Check if there are results
if 'results' in data:
    print(f"\nFound {len(data['results'])} results:")
    for i, result in enumerate(data['results']):
        print(f"\nResult {i+1}:")
        print(json.dumps(result, indent=2))
elif 'data' in data:
    print(f"\nFound {len(data['data'])} data items:")
    for i, item in enumerate(data['data']):
        print(f"\nData item {i+1}:")
        print(json.dumps(item, indent=2))
else:
    print("No results found in the response data:")
    print(json.dumps(data, indent=2))

Using graph ID: g-4o5cjg3ix7
Query results:
{
  "results": [
    {
      "p": [
        {
          "~id": "3fb5903d1c30601242b0cdfaefbcdcac",
          "~entityType": "node",
          "~labels": [
            "__Statement__"
          ],
          "~properties": {
            "details": "",
            "value": "Neptune Database can scale to 100,000 queries per second and provides Multi-AZ high availability and multi-Region deployments."
          }
        }
      ]
    },
    {
      "p": [
        {
          "~id": "f0f65effa6cb84dffae78d6a94ac458a",
          "~entityType": "node",
          "~labels": [
            "__Statement__"
          ],
          "~properties": {
            "details": "",
            "value": "Neptune Analytics stores large graph datasets in memory to enable quick analysis and insights."
          }
        }
      ]
    },
    {
      "p": [
        {
          "~id": "971b5a6ec975bfd78b42429a7d1ced83",
          "~entityType": "node",
          "~labe

In [20]:
import boto3
import json
import os
from urllib.parse import urlparse

# Initialize Neptune client
neptune = boto3.client('neptune-graph', region_name='us-east-1')

# Get graph ID from environment
graph_store_url = os.environ['GRAPH_STORE']
graph_id = urlparse(graph_store_url).netloc

print(f"Exploring graph: {graph_id}")

# Debug: Let's first try a simple query to see the response structure
def debug_response_structure():
    """Debug the response structure from Neptune Analytics."""
    query = """
    MATCH (n)
    RETURN count(n) as total_nodes
    LIMIT 1
    """
    
    try:
        response = neptune.execute_query(
            graphIdentifier=graph_id,
            queryString=query,
            language='opencypher'
        )
        
        print("Response keys:", response.keys())
        print("Response structure:")
        print(json.dumps(response, indent=2, default=str))
        
        # Check if payload exists and read it
        if 'payload' in response:
            payload = response['payload']
            print(f"\nPayload type: {type(payload)}")
            
            # Try to read the payload
            try:
                data = json.loads(payload.read().decode('utf-8'))
                print("Parsed payload data:")
                print(json.dumps(data, indent=2))
                
                # Check what keys are in the data
                if isinstance(data, dict):
                    print(f"Data keys: {data.keys()}")
                    if 'results' in data:
                        print(f"Number of results: {len(data['results'])}")
                        if data['results']:
                            print("First result structure:")
                            print(json.dumps(data['results'][0], indent=2))
                
            except Exception as e:
                print(f"Error parsing payload: {e}")
                print(f"Payload content: {payload.read()}")
        
    except Exception as e:
        print(f"Error executing query: {e}")
        print(f"Full error details: {type(e).__name__}: {str(e)}")

# Run the debug function
debug_response_structure()

Exploring graph: g-4o5cjg3ix7
Response keys: dict_keys(['ResponseMetadata', 'payload'])
Response structure:
{
  "ResponseMetadata": {
    "RequestId": "bbc89db5-a9b1-4d34-a3ff-c7074adc7d2d",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "transfer-encoding": "chunked",
      "content-type": "application/json;charset=UTF-8",
      "x-amzn-requestid": "bbc89db5-a9b1-4d34-a3ff-c7074adc7d2d"
    },
    "RetryAttempts": 0
  },
  "payload": "<botocore.response.StreamingBody object at 0x31c509870>"
}

Payload type: <class 'botocore.response.StreamingBody'>
Parsed payload data:
{
  "results": [
    {
      "total_nodes": 191
    }
  ]
}
Data keys: dict_keys(['results'])
Number of results: 1
First result structure:
{
  "total_nodes": 191
}


In [23]:
import boto3
import json
import os
from urllib.parse import urlparse

# Initialize Neptune client
neptune = boto3.client('neptune-graph', region_name='us-east-1')

# Get graph ID from environment
graph_store_url = os.environ['GRAPH_STORE']
graph_id = urlparse(graph_store_url).netloc

print(f"Exploring graph: {graph_id}")

def execute_query(query, parameters=None):
    """Helper function to execute queries and parse responses."""
    try:
        # Ensure parameters is always a dict, even if None
        if parameters is None:
            parameters = {}
        
        response = neptune.execute_query(
            graphIdentifier=graph_id,
            queryString=query,
            parameters=parameters,
            language='opencypher'
        )
        
        payload = response['payload']
        data = json.loads(payload.read().decode('utf-8'))
        return data['results']
        
    except Exception as e:
        print(f"Error executing query: {e}")
        return []

# 1. Count all node types
def count_node_types():
    """Count all different types of nodes in the graph."""
    query = """
    MATCH (n)
    RETURN labels(n) as node_type, count(n) as count
    ORDER BY count DESC
    """
    
    results = execute_query(query)
    
    print(" Node Counts:")
    for result in results:
        node_type = result['node_type']
        count = result['count']
        print(f"  {node_type}: {count}")
    
    return results

# 2. Count all relationship types
def count_relationship_types():
    """Count all different types of relationships in the graph."""
    query = """
    MATCH ()-[r]->()
    RETURN type(r) as relationship_type, count(r) as count
    ORDER BY count DESC
    """
    
    results = execute_query(query)
    
    print("\n Relationship Counts:")
    for result in results:
        rel_type = result['relationship_type']
        count = result['count']
        print(f"  {rel_type}: {count}")
    
    return results

# 3. Show sample sources
def show_sources():
    """Show all sources in the graph."""
    query = """
    MATCH (s:`__Source__`)
    RETURN s.sourceId as source_id, s.metadata as metadata
    LIMIT 5
    """
    
    results = execute_query(query)
    
    print("\n Sources:")
    for result in results:
        source_id = result['source_id']
        metadata = result['metadata']
        print(f"  Source ID: {source_id}")
        print(f"    URL: {metadata.get('url', 'N/A')}")
        print(f"    Source: {metadata.get('source', 'N/A')}")

# 4. Show sample statements with their facts
def show_statements_with_facts():
    """Show statements and their supporting facts."""
    query = """
    MATCH (s:`__Statement__`)<-[:`__SUPPORTS__`]-(f:`__Fact__`)
    RETURN s.statement as statement, collect(f.fact) as facts
    LIMIT 5
    """
    
    results = execute_query(query)
    
    print("\n💬 Statements with Supporting Facts:")
    for result in results:
        statement = result['statement']
        facts = result['facts']
        print(f"  Statement: {statement}")
        print(f"    Facts: {facts}")
        print()

# 5. Show topics and their statements
def show_topics_and_statements():
    """Show topics and the statements they contain."""
    query = """
    MATCH (t:`__Topic__`)<--(s:`__Statement__`)
    RETURN t.topic as topic, collect(s.statement) as statements
    LIMIT 5
    """
    
    results = execute_query(query)
    
    print("\n📋 Topics and Statements:")
    for result in results:
        topic = result['topic']
        statements = result['statements']
        print(f"  Topic: {topic}")
        print(f"    Statements ({len(statements)}):")
        for i, statement in enumerate(statements[:3]):  # Show first 3 statements
            print(f"      {i+1}. {statement}")
        if len(statements) > 3:
            print(f"      ... and {len(statements) - 3} more")
        print()

# 6. Show graph structure for a specific topic
def show_topic_structure(topic_name="Neptune Analytics"):
    """Show the complete structure for a specific topic."""
    query = """
    MATCH (s:`__Source__`)<--(c:`__Chunk__`)<--(t:`__Topic__`)<--(st:`__Statement__`)
    WHERE t.topic = $topic_name
    RETURN s.sourceId as source_id, t.topic as topic, count(st) as statement_count
    """
    
    results = execute_query(query, {'topic_name': topic_name})
    
    print(f"\n🏗️  Graph Structure for Topic: {topic_name}")
    if results:
        for result in results:
            source_id = result['source_id']
            topic = result['topic']
            count = result['statement_count']
            print(f"  Source: {source_id}")
            print(f"    Topic: {topic}")
            print(f"    Statements: {count}")
    else:
        print(f"  No results found for topic: {topic_name}")

# 7. Show cross-references between sources
def show_cross_references():
    """Show how facts connect statements across different sources."""
    query = """
    MATCH (s1:`__Source__`)<--(:`__Chunk__`)<--(:`__Topic__`)<--(st1:`__Statement__`)
           <-[:`__SUPPORTS__`]-(f:`__Fact__`)-[:`__SUPPORTS__`]->(st2:`__Statement__`)
           -->(:`__Topic__`)-->(:`__Chunk__`)-->(s2:`__Source__`)
    WHERE s1 <> s2
    RETURN s1.sourceId as source1, s2.sourceId as source2, 
           st1.statement as statement1, st2.statement as statement2,
           f.fact as connecting_fact
    LIMIT 3
    """
    
    results = execute_query(query)
    
    print("\n🔗 Cross-References Between Sources:")
    if results:
        for result in results:
            source1 = result['source1']
            source2 = result['source2']
            statement1 = result['statement1']
            statement2 = result['statement2']
            fact = result['connecting_fact']
            print(f"  {source1} ↔ {source2}")
            print(f"    Via fact: {fact}")
            print(f"    Statement 1: {statement1}")
            print(f"    Statement 2: {statement2}")
            print()
    else:
        print("  No cross-references found")

# 8. Show total counts
def show_total_counts():
    """Show total counts of nodes and relationships."""
    query = """
    MATCH (n)
    RETURN count(n) as total_nodes
    """
    
    results = execute_query(query)
    total_nodes = results[0]['total_nodes'] if results else 0
    
    query = """
    MATCH ()-[r]->()
    RETURN count(r) as total_relationships
    """
    
    results = execute_query(query)
    total_relationships = results[0]['total_relationships'] if results else 0
    
    print(f"\n📊 Total Graph Statistics:")
    print(f"  Total Nodes: {total_nodes}")
    print(f"  Total Relationships: {total_relationships}")

# 9. Let's also check if there are any nodes at all
def check_if_graph_has_data():
    """Check if the graph actually has any data."""
    query = """
    MATCH (n)
    RETURN count(n) as node_count
    """
    
    results = execute_query(query)
    node_count = results[0]['node_count'] if results else 0
    
    if node_count == 0:
        print("\n⚠️  WARNING: The graph appears to be empty!")
        print("This could mean:")
        print("  1. The build process didn't complete successfully")
        print("  2. The graph was cleared or reset")
        print("  3. There's an issue with the graph connection")
        print("\nRecommendations:")
        print("  1. Re-run the 02-Separate-Extract-and-Build.ipynb notebook")
        print("  2. Check if the build process completed successfully")
        print("  3. Verify the graph ID is correct")
    else:
        print(f"\n✅ Graph has {node_count} nodes")

# Run all explorations
print("🔍 Exploring your GraphRAG knowledge graph...\n")

try:
    check_if_graph_has_data()
    show_total_counts()
    count_node_types()
    count_relationship_types()
    show_sources()
    show_topics_and_statements()
    show_statements_with_facts()
    show_topic_structure()
    show_cross_references()
    
    print("\n✅ Graph exploration complete!")
    
except Exception as e:
    print(f"❌ Error exploring graph: {e}")
    print("Make sure your Neptune Analytics graph is active and accessible.")

Exploring graph: g-4o5cjg3ix7
🔍 Exploring your GraphRAG knowledge graph...


✅ Graph has 191 nodes

📊 Total Graph Statistics:
  Total Nodes: 191
  Total Relationships: 417
 Node Counts:
  ['__Fact__']: 83
  ['__Statement__']: 35
  ['__Entity__']: 26
  ['__SYS_SV__EntityClassification__']: 17
  ['__SYS_SV__StatementTopic__']: 8
  ['__Topic__']: 8
  ['__Chunk__']: 6
  ['__SYS_Class__']: 6
  ['__Source__']: 2

 Relationship Counts:
  __SUPPORTS__: 85
  __SUBJECT__: 83
  __NEXT__: 61
  __MENTIONED_IN__: 43
  __BELONGS_TO__: 35
  __PREVIOUS__: 32
  __RELATION__: 25
  __OBJECT__: 25
  __SYS_RELATION__: 22
  __EXTRACTED_FROM__: 6

 Sources:
  Source ID: None
❌ Error exploring graph: 'NoneType' object has no attribute 'get'
Make sure your Neptune Analytics graph is active and accessible.


#### Metadata filtering

In [ ]:
%reload_ext dotenv
%dotenv

import os

from graphrag_toolkit.lexical_graph import set_logging_config
from graphrag_toolkit.lexical_graph import LexicalGraphQueryEngine
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.lexical_graph.storage import VectorStoreFactory
from graphrag_toolkit.lexical_graph.metadata import FilterConfig

from llama_index.core.vector_stores.types import FilterOperator, MetadataFilter

set_logging_config('INFO')

graph_store = GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE'])
vector_store = VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE'])

query_engine = LexicalGraphQueryEngine.for_traversal_based_search(
    graph_store, 
    vector_store,
    filter_config = FilterConfig(
        MetadataFilter(
            key='url',
            value='https://docs.aws.amazon.com/neptune/latest/userguide/intro.html',
            operator=FilterOperator.EQ
        )
    )
)

response = query_engine.query("What are the differences between Neptune Database and Neptune Analytics?")

print(f"""{response.response}

retrieve_ms: {int(response.metadata['retrieve_ms'])}
answer_ms  : {int(response.metadata['answer_ms'])}
total_ms   : {int(response.metadata['total_ms'])}
""")

In [ ]:
for n in response.source_nodes:
    print(n.text)

#### Set subretriever

In the example below, the `TraversalBasedRetriever` is configured with a `ChunkBasedSearch` subretriever. (You can also try with `EntityBasedSearch` and `EntityContextSearch`).

In [ ]:
%reload_ext dotenv
%dotenv

import os

from graphrag_toolkit.lexical_graph import LexicalGraphQueryEngine
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.lexical_graph.storage import VectorStoreFactory
from graphrag_toolkit.lexical_graph.retrieval.retrievers import ChunkBasedSearch
from graphrag_toolkit.lexical_graph.retrieval.retrievers import EntityBasedSearch
from graphrag_toolkit.lexical_graph.retrieval.retrievers import EntityContextSearch

graph_store = GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE'])
vector_store = VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE'])

query_engine = LexicalGraphQueryEngine.for_traversal_based_search(
    graph_store, 
    vector_store,
    retrievers=[ChunkBasedSearch]
)

response = query_engine.query("What are the differences between Neptune Database and Neptune Analytics?")

print(f"""{response.response}

retrieve_ms: {int(response.metadata['retrieve_ms'])}
answer_ms  : {int(response.metadata['answer_ms'])}
total_ms   : {int(response.metadata['total_ms'])}
""")